In [2]:

from pathlib import Path
import xml.etree.ElementTree as ET

import pandas as pd
import numpy as np
from IPython.display import display

# Define the directory containing the original datasets
DATA_DIR = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Project_Data"
)

USERS_PATH = DATA_DIR / "users.xml"

# Verify that the file exists
if not USERS_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {USERS_PATH}"
    )

print("Dataset path:", USERS_PATH)
print(
    "File size (MB):",
    round(USERS_PATH.stat().st_size / (1024 ** 2), 2)
)

# Parse the XML incrementally to reduce memory usage
records = []

context = ET.iterparse(
    USERS_PATH,
    events=("start", "end")
)

_, root = next(context)

for event, elem in context:
    if event == "end" and elem.tag == "row":

        record = {
            child.tag: child.text
            for child in elem
        }

        records.append(record)

        # Release processed XML elements from memory
        root.clear()

# Convert extracted records into a DataFrame
df_users = pd.DataFrame.from_records(records)

# Preserve the original dataset for comparison
df_users_raw = df_users.copy()

print("\nDataset shape:", df_users.shape)

print("\nColumn names:")
print(df_users.columns.tolist())

display(df_users.head())


Dataset path: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\users.xml
File size (MB): 216.98

Dataset shape: (487292, 9)

Column names:
['id_user', 'timestamp_user_creation', 'username', 'follower_count', 'following_count', 'profile_url', 'finished_follower_scrape', 'finished_following_scrape', 'finished_truth_scrape']


,id_user,timestamp_user_creation,username,follower_count,following_count,profile_url,finished_follower_scrape,finished_following_scrape,finished_truth_scrape
0,1238,2022-02-01 00:00:00,jeffgolfs,886,204,https://truthsocial.com/@jeffgolfs,f,f,f
1,50,2022-02-01 00:00:00,MizW,2365,2929,https://truthsocial.com/@MizW,f,f,f
2,6951,2022-02-01 00:00:00,SusanCreech,870,1232,https://truthsocial.com/@SusanCreech,f,f,f
3,9861,2022-02-01 00:00:00,BMF6FeeTDeeP,1086,1654,https://truthsocial.com/@BMF6FeeTDeeP,f,f,f
4,34,2022-02-01 00:00:00,DC_Draino,286146,81,https://truthsocial.com/@DC_Draino,t,t,t


In [3]:

# Define a summary of initial data quality
quality_baseline = pd.DataFrame({
    "dtype": df_users.dtypes.astype(str),
    "null_count": df_users.isna().sum(),
    "null_percentage": (
        df_users.isna().mean() * 100
    ).round(2),
    "sentinel_count": [
        df_users[col].astype(str).str.strip().eq("-1").sum()
        for col in df_users.columns
    ],
    "unique_values": df_users.nunique(dropna=True)
})

# Calculate the percentage of sentinel values
quality_baseline["sentinel_percentage"] = (
    quality_baseline["sentinel_count"]
    / len(df_users) * 100
).round(2)

display(quality_baseline)

print("\nTotal records:", len(df_users))
print("Duplicated user IDs:",
      df_users["id_user"].duplicated().sum())
print("Completely duplicated records:",
      df_users.duplicated().sum())


,dtype,null_count,null_percentage,sentinel_count,unique_values,sentinel_percentage
id_user,str,0,0.0,0,487292,0.00
timestamp_user_creation,str,0,0.0,465337,16,95.49
username,str,0,0.0,18,487153,0.00
follower_count,str,0,0.0,465337,8695,95.49
following_count,str,0,0.0,465337,7313,95.49
profile_url,str,0,0.0,0,487150,0.00
finished_follower_scrape,str,0,0.0,465337,3,95.49
finished_following_scrape,str,0,0.0,465337,3,95.49
finished_truth_scrape,str,0,0.0,465337,3,95.49



Total records: 487292
Duplicated user IDs: 0
Completely duplicated records: 0


In [4]:

# Identify columns containing sentinel values
sentinel_columns = [
    "timestamp_user_creation",
    "follower_count",
    "following_count",
    "finished_follower_scrape",
    "finished_following_scrape",
    "finished_truth_scrape"
]

# Build a boolean matrix identifying sentinel values
sentinel_mask = df_users[sentinel_columns].eq("-1")

# Count sentinel values per user
df_users["sentinel_count"] = sentinel_mask.sum(axis=1)

print("Distribution of sentinel values per user:")
display(
    df_users["sentinel_count"]
    .value_counts()
    .sort_index()
    .rename_axis("sentinel_count")
    .reset_index(name="number_of_users")
)

# Analyze combinations of missing attributes
missing_patterns = (
    sentinel_mask
    .value_counts()
    .reset_index(name="number_of_users")
    .sort_values("number_of_users", ascending=False)
)

missing_patterns["percentage"] = (
    missing_patterns["number_of_users"]
    / len(df_users) * 100
).round(2)

print("\nMost frequent missing-value patterns:")
display(missing_patterns.head(10))

# Verify whether sentinel values always occur together
all_missing = sentinel_mask.all(axis=1)
none_missing = ~sentinel_mask.any(axis=1)
partial_missing = ~(all_missing | none_missing)

print("\nUsers with all six sentinel values:", all_missing.sum())
print("Users with no sentinel values:", none_missing.sum())
print("Users with partial sentinel values:", partial_missing.sum())


Distribution of sentinel values per user:


,sentinel_count,number_of_users
0,0,21955
1,6,465337



Most frequent missing-value patterns:


,timestamp_user_creation,follower_count,following_count,finished_follower_scrape,finished_following_scrape,finished_truth_scrape,number_of_users,percentage
0,True,True,True,True,True,True,465337,95.49
1,False,False,False,False,False,False,21955,4.51



Users with all six sentinel values: 465337
Users with no sentinel values: 21955
Users with partial sentinel values: 0


In [5]:

# Classify records based on sentinel values
df_users["data_completeness"] = np.select(
    [
        df_users["sentinel_count"].eq(0),
        df_users["sentinel_count"].eq(6)
    ],
    [
        "Complete",
        "Incomplete"
    ],
    default="Partially incomplete"
)

# Summarize the groups
completeness_summary = (
    df_users.groupby("data_completeness")
    .agg(
        total_users=("id_user", "size"),
        unique_user_ids=("id_user", "nunique"),
        unique_usernames=("username", "nunique"),
        unique_profile_urls=("profile_url", "nunique")
    )
)

completeness_summary["percentage"] = (
    completeness_summary["total_users"]
    / len(df_users) * 100
).round(2)

display(completeness_summary)

# Display examples from incomplete records
print("\nExamples of incomplete users:")
display(
    df_users.loc[
        df_users["data_completeness"] == "Incomplete"
    ].head(10)
)


,total_users,unique_user_ids,unique_usernames,unique_profile_urls,percentage
data_completeness,,,,,
Complete,21955,21955,21955,21955,4.51
Incomplete,465337,465337,465198,465198,95.49



Examples of incomplete users:


,id_user,timestamp_user_creation,username,follower_count,following_count,profile_url,finished_follower_scrape,finished_following_scrape,finished_truth_scrape,sentinel_count,data_completeness
21955,22025,-1,hazel_eyez,-1,-1,https://truthsocial.com/@hazel_eyez,-1,-1,-1,6,Incomplete
21956,22026,-1,Trucker55,-1,-1,https://truthsocial.com/@Trucker55,-1,-1,-1,6,Incomplete
21957,22027,-1,foxxed,-1,-1,https://truthsocial.com/@foxxed,-1,-1,-1,6,Incomplete
21958,22028,-1,RyderMcCue,-1,-1,https://truthsocial.com/@RyderMcCue,-1,-1,-1,6,Incomplete
21959,22029,-1,TruePatriotNurse21,-1,-1,https://truthsocial.com/@TruePatriotNurse21,-1,-1,-1,6,Incomplete
21960,22030,-1,myteam,-1,-1,https://truthsocial.com/@myteam,-1,-1,-1,6,Incomplete
21961,22031,-1,JB2ndID,-1,-1,https://truthsocial.com/@JB2ndID,-1,-1,-1,6,Incomplete
21962,22032,-1,ConstitutionalGranny,-1,-1,https://truthsocial.com/@ConstitutionalGranny,-1,-1,-1,6,Incomplete
21963,22033,-1,ArtLady17,-1,-1,https://truthsocial.com/@ArtLady17,-1,-1,-1,6,Incomplete
21964,22034,-1,Debfea,-1,-1,https://truthsocial.com/@Debfea,-1,-1,-1,6,Incomplete


In [6]:

# Convert user IDs to numeric for analysis
df_users["id_user_numeric"] = pd.to_numeric(
    df_users["id_user"],
    errors="coerce"
)

# Compare user ID distributions across completeness groups
id_summary = (
    df_users.groupby("data_completeness")
    ["id_user_numeric"]
    .agg(["count", "min", "median", "max"])
)

print("User ID distribution by completeness group:")
display(id_summary)

# Check completeness distribution before and after the
# boundary observed in the dataset
boundary = 22025

df_users["id_range"] = np.where(
    df_users["id_user_numeric"] < boundary,
    "ID < 22025",
    "ID >= 22025"
)

id_range_summary = pd.crosstab(
    df_users["id_range"],
    df_users["data_completeness"]
)

print("\nCompleteness by ID range:")
display(id_range_summary)

# Check whether complete and incomplete records
# appear in separate blocks in the original XML
completeness_transitions = (
    df_users["data_completeness"]
    .ne(df_users["data_completeness"].shift())
)

print(
    "\nNumber of completeness blocks:",
    completeness_transitions.sum()
)

print("\nStarting positions of completeness blocks:")
display(
    df_users.loc[
        completeness_transitions,
        ["id_user", "data_completeness"]
    ].head(20)
)


User ID distribution by completeness group:


,count,min,median,max
data_completeness,,,,
Complete,21955,22,11047.0,22024
Incomplete,465337,22025,254693.0,487361



Completeness by ID range:


data_completeness,Complete,Incomplete
id_range,,
ID < 22025,21955,0
ID >= 22025,0,465337



Number of completeness blocks: 2

Starting positions of completeness blocks:


,id_user,data_completeness
0,1238,Complete
21955,22025,Incomplete


In [7]:

# Define the path to the follows dataset
FOLLOWS_PATH = DATA_DIR / "follows.json"

if not FOLLOWS_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {FOLLOWS_PATH}"
    )

print("Follows dataset path:", FOLLOWS_PATH)

print(
    "File size (MB):",
    round(FOLLOWS_PATH.stat().st_size / (1024 ** 2), 2)
)

# Load follows dataset
df_follows = pd.read_json(FOLLOWS_PATH)

print("\nDataset shape:", df_follows.shape)

print("\nColumn names:")
print(df_follows.columns.tolist())

print("\nData types:")
display(df_follows.dtypes)

print("\nFirst five records:")
display(df_follows.head())


Follows dataset path: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\follows.json
File size (MB): 117.6

Dataset shape: (4002115, 2)

Column names:
['follower', 'followed']

Data types:


follower    int64
followed    int64
dtype: object


First five records:


,follower,followed
0,12475,22025
1,12475,22026
2,12475,22027
3,12475,22028
4,12475,17280


In [8]:

# Normalize user identifiers for cross-dataset comparison
user_ids = pd.to_numeric(
    df_users["id_user"],
    errors="coerce"
)

follower_ids = pd.to_numeric(
    df_follows["follower"],
    errors="coerce"
)

followed_ids = pd.to_numeric(
    df_follows["followed"],
    errors="coerce"
)

# Identify all unique users appearing in follows.json
network_users = pd.Index(
    pd.concat(
        [follower_ids, followed_ids],
        ignore_index=True
    ).dropna().unique()
)

# Check whether each user appears in the follows network
df_users["appears_in_follows"] = (
    user_ids.isin(network_users)
)

# Calculate coverage for each completeness group
coverage_summary = (
    df_users.groupby("data_completeness")
    ["appears_in_follows"]
    .agg(["sum", "count", "mean"])
)

coverage_summary.columns = [
    "users_in_follows",
    "total_users",
    "coverage_rate"
]

coverage_summary["coverage_percentage"] = (
    coverage_summary["coverage_rate"] * 100
).round(2)

display(coverage_summary)

print("\nTotal unique users in follows.json:",
      len(network_users))

print(
    "Users from users.xml appearing in follows.json:",
    df_users["appears_in_follows"].sum()
)

print(
    "Users from users.xml absent from follows.json:",
    (~df_users["appears_in_follows"]).sum()
)

print(
    "Network users absent from users.xml:",
    (~network_users.isin(user_ids.dropna())).sum()
)


,users_in_follows,total_users,coverage_rate,coverage_percentage
data_completeness,,,,
Complete,21801,21955,0.992986,99.30
Incomplete,423149,465337,0.909339,90.93



Total unique users in follows.json: 444951
Users from users.xml appearing in follows.json: 444950
Users from users.xml absent from follows.json: 42342
Network users absent from users.xml: 1


In [9]:

# Check the integrity of follow relationships
duplicate_edges = df_follows.duplicated(
    subset=["follower", "followed"]
).sum()

self_follows = (
    df_follows["follower"] == df_follows["followed"]
).sum()

print("Total relationships:", len(df_follows))
print("Duplicated relationships:", duplicate_edges)
print("Self-follow relationships:", self_follows)

# Create a separate working copy of valid relationships
df_follows_clean = (
    df_follows
    .dropna(subset=["follower", "followed"])
    .drop_duplicates(subset=["follower", "followed"])
    .copy()
)

# Exclude self-follow relationships from degree calculations
df_follows_clean = df_follows_clean.loc[
    df_follows_clean["follower"]
    != df_follows_clean["followed"]
]

# Count distinct observed followers for each user
observed_followers = (
    df_follows_clean
    .groupby("followed")["follower"]
    .nunique()
    .rename("observed_follower_count")
)

# Count distinct observed following accounts for each user
observed_following = (
    df_follows_clean
    .groupby("follower")["followed"]
    .nunique()
    .rename("observed_following_count")
)

# Create a comparison dataset without modifying original values
df_network_comparison = df_users[
    ["id_user", "data_completeness",
     "follower_count", "following_count"]
].copy()

df_network_comparison["id_user"] = pd.to_numeric(
    df_network_comparison["id_user"],
    errors="coerce"
)

df_network_comparison = df_network_comparison.merge(
    observed_followers,
    left_on="id_user",
    right_index=True,
    how="left"
)

df_network_comparison = df_network_comparison.merge(
    observed_following,
    left_on="id_user",
    right_index=True,
    how="left"
)

# Missing network degrees mean no relationships were observed
df_network_comparison[
    ["observed_follower_count", "observed_following_count"]
] = df_network_comparison[
    ["observed_follower_count", "observed_following_count"]
].fillna(0).astype("int64")

print("\nNetwork count comparison:")
display(df_network_comparison.head(10))

print("\nObserved network counts for incomplete users:")
display(
    df_network_comparison.loc[
        df_network_comparison["data_completeness"] == "Incomplete",
        ["observed_follower_count", "observed_following_count"]
    ].describe()
)


Total relationships: 4002115
Duplicated relationships: 99
Self-follow relationships: 0

Network count comparison:


,id_user,data_completeness,follower_count,following_count,observed_follower_count,observed_following_count
0,1238,Complete,886,204,17,0
1,50,Complete,2365,2929,81,1
2,6951,Complete,870,1232,17,0
3,9861,Complete,1086,1654,21,0
4,34,Complete,286146,81,981,87
5,40,Complete,1222,1402,63,1456
6,37,Complete,68063,21,262,22
7,22,Complete,455386,2,822,2
8,25,Complete,6942,221,53,0
9,52,Complete,1900,2131,71,1



Observed network counts for incomplete users:


,observed_follower_count,observed_following_count
count,465337.000000,465337.000000
mean,5.484232,0.005256
std,13.491696,2.602313
min,0.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,0.000000
75%,4.000000,0.000000
max,539.000000,1680.000000


In [10]:

# Select users with available original profile counts
complete_users = df_network_comparison.loc[
    df_network_comparison["data_completeness"] == "Complete"
].copy()

# Convert original counts to numeric
complete_users["follower_count"] = pd.to_numeric(
    complete_users["follower_count"],
    errors="coerce"
)

complete_users["following_count"] = pd.to_numeric(
    complete_users["following_count"],
    errors="coerce"
)

# Calculate differences between observed and original counts
complete_users["follower_difference"] = (
    complete_users["observed_follower_count"]
    - complete_users["follower_count"]
)

complete_users["following_difference"] = (
    complete_users["observed_following_count"]
    - complete_users["following_count"]
)

# Evaluate exact matches and inconsistencies
validation_summary = pd.DataFrame({
    "Metric": ["Followers", "Following"],
    "Exact matches": [
        complete_users["follower_difference"].eq(0).sum(),
        complete_users["following_difference"].eq(0).sum()
    ],
    "Observed lower": [
        complete_users["follower_difference"].lt(0).sum(),
        complete_users["following_difference"].lt(0).sum()
    ],
    "Observed higher": [
        complete_users["follower_difference"].gt(0).sum(),
        complete_users["following_difference"].gt(0).sum()
    ]
})

for col in ["Exact matches", "Observed lower", "Observed higher"]:
    validation_summary[col + " (%)"] = (
        validation_summary[col]
        / len(complete_users) * 100
    ).round(2)

display(validation_summary)

# Summarize discrepancies
print("\nFollower count differences:")
display(complete_users["follower_difference"].describe())

print("\nFollowing count differences:")
display(complete_users["following_difference"].describe())


,Metric,Exact matches,Observed lower,Observed higher,Exact matches (%),Observed lower (%),Observed higher (%)
0,Followers,99,21308,548,0.45,97.05,2.50
1,Following,2251,19179,525,10.25,87.36,2.39



Follower count differences:


count    2.195500e+04
mean    -7.245762e+03
std      5.626274e+04
min     -4.025987e+06
25%     -5.052000e+03
50%     -1.888000e+03
75%     -4.030000e+02
max      7.680000e+02
Name: follower_difference, dtype: float64


Following count differences:


count    21955.000000
mean     -3175.333318
std       5337.241052
min     -99514.000000
25%      -5051.500000
50%      -1270.000000
75%        -64.000000
max       2553.000000
Name: following_difference, dtype: float64